## 4 .Провести предварительный анализ, очистку и слияние данных. Этот этап включает в себя вывод информации о количественных характеристиках датасета, информацию об отсутствующих значениях, характеристиках и физическом смысле каждого атрибута данных, его значимости для проверки гипотезы, вывод нескольких точек данных для иллюстрации структуры данных. Выполнить аннотирование и векторизацию неструктурированных данных (аудио, видео, текст). С целью слияния необходимо определить типы шкал, аналитические роли и общие атрибуты (агрегаты данных) в разных форматах.

In [12]:
import os, glob, json, time
import xml.etree.ElementTree as ET
import pandas as pd

T = {}   # замеры времени для п. 2.5

# 1. Lending Club (реляционный)
t0 = time.perf_counter()
cols = ["loan_amnt", "term", "int_rate", "installment", "grade", "emp_length",
        "home_ownership", "annual_inc", "verification_status", "issue_d",
        "loan_status", "purpose", "addr_state", "dti", "delinq_2yrs",
        "earliest_cr_line", "fico_range_low", "fico_range_high",
        "open_acc", "pub_rec", "revol_bal", "revol_util"]
lc = pd.read_csv("accepted_2007_to_2018Q4.csv", usecols=cols, low_memory=False)
lc = lc.dropna(subset=["loan_status"])
lc["issue_month"] = pd.to_datetime(lc["issue_d"], format="%b-%Y")
T["Lending Club (CSV)"] = time.perf_counter() - t0

# 2. FRED безработица (XML): читаем сохранённые файлы data/raw/xml/{ST}UR.xml
t0 = time.perf_counter()
rows = []
for path in sorted(glob.glob("data/raw/xml/*UR.xml")):
    st = os.path.basename(path)[:2]
    for o in ET.parse(path).getroot().findall("observation"):
        v = o.get("value")
        rows.append({"state": st, "date": o.get("date"),
                     "unemployment": None if v == "." else float(v)})
df_unemp = pd.DataFrame(rows)
df_unemp["date"] = pd.to_datetime(df_unemp["date"])
T["FRED безработица (XML)"] = time.perf_counter() - t0

# 3. FRED ставка ФРС (JSON)
t0 = time.perf_counter()
with open("FEDFUNDS.json", encoding="utf-8") as f:
    raw = json.load(f)
ff = pd.json_normalize(raw["observations"])[["date", "value"]]
ff["date"] = pd.to_datetime(ff["date"])
ff["fed_funds_rate"] = pd.to_numeric(ff["value"], errors="coerce")
ff = ff.drop(columns="value")
ff = ff[(ff["date"] >= "2005-01-01") & (ff["date"] <= "2019-12-01")].reset_index(drop=True)
ff["chg_3m"] = ff["fed_funds_rate"].diff(3)
ff["chg_12m"] = ff["fed_funds_rate"].diff(12)
ff["ma_6m"] = ff["fed_funds_rate"].rolling(6).mean()
T["FRED ставка (JSON)"] = time.perf_counter() - t0

# 4. CFPB (текст)
t0 = time.perf_counter()
ccols = ["Date received", "Product", "Sub-product", "Issue", "Sub-issue",
         "Consumer complaint narrative", "Company", "State", "Submitted via",
         "Date sent to company", "Company response to consumer", "Timely response?", "Complaint ID"]
parts = []
for ch in pd.read_csv("CCDB_Export_1_December_2011_through_April_2018.csv",
                      usecols=ccols, chunksize=200_000, low_memory=False):
    parts.append(ch.dropna(subset=["Consumer complaint narrative"]))
df_txt = pd.concat(parts, ignore_index=True)
df_txt["Date received"] = pd.to_datetime(df_txt["Date received"], errors="coerce")
df_txt["Date sent to company"] = pd.to_datetime(df_txt["Date sent to company"], errors="coerce")
T["CFPB (CSV, чтение частями)"] = time.perf_counter() - t0

print(pd.Series(T).round(1).rename("секунд"))
print("lc:", lc.shape, "| unemp:", df_unemp.shape, "| ff:", ff.shape, "| txt:", df_txt.shape)

Lending Club (CSV)            29.5
FRED безработица (XML)         0.7
FRED ставка (JSON)             0.0
CFPB (CSV, чтение частями)     8.6
Name: секунд, dtype: float64
lc: (2260668, 23) | unemp: (8568, 3) | ff: (180, 5) | txt: (291574, 13)


### 4.1	Количественные характеристики, пропуски, словарь атрибутов (смысл, шкала, роль), примеры строк

In [14]:
import time
import pandas as pd
from IPython.display import display

def profile(df, name, n_sample=3):
    t0 = time.perf_counter()
    print(name)
    print("Строк × колонок:", df.shape,
          "| память, МБ:", round(df.memory_usage(deep=True).sum() / 1e6, 1))
    prof = pd.DataFrame({"dtype": df.dtypes.astype(str),
                         "пропуски_%": (df.isna().mean() * 100).round(2),
                         "уникальных": df.nunique()})
    display(prof)
    num = df.select_dtypes("number")
    if not num.empty:
        display(num.describe().T.round(2))
    display(df.sample(n_sample, random_state=1))
    print(f"Время профилирования: {time.perf_counter() - t0:.1f} c")

profile(lc, "Lending Club (реляционный)")
profile(df_unemp, "FRED: безработица по штатам (XML)")
profile(ff, "FRED: ставка ФРС (JSON)")
profile(df_txt.drop(columns=["Consumer complaint narrative"]), "CFPB: метаданные жалоб (текст)")

Lending Club (реляционный)
Строк × колонок: (2260668, 23) | память, МБ: 1534.9


,dtype,пропуски_%,уникальных
loan_amnt,float64,0.00,1572
term,object,0.00,2
int_rate,float64,0.00,673
installment,float64,0.00,93301
grade,object,0.00,7
emp_length,object,6.50,11
home_ownership,object,0.00,6
annual_inc,float64,0.00,89368
verification_status,object,0.00,3
issue_d,object,0.00,139


,count,mean,std,min,25%,50%,75%,max
loan_amnt,2260668.0,15046.93,9190.25,500.00,8000.00,12900.00,20000.00,4.000000e+04
int_rate,2260668.0,13.09,4.83,5.31,9.49,12.62,15.99,3.099000e+01
installment,2260668.0,445.81,267.17,4.93,251.65,377.99,593.32,1.719830e+03
annual_inc,2260664.0,77992.43,112696.20,0.00,46000.00,65000.00,93000.00,1.100000e+08
dti,2258957.0,18.82,14.18,-1.00,11.89,17.84,24.49,9.990000e+02
delinq_2yrs,2260639.0,0.31,0.87,0.00,0.00,0.00,0.00,5.800000e+01
fico_range_low,2260668.0,698.59,33.01,610.00,675.00,690.00,715.00,8.450000e+02
fico_range_high,2260668.0,702.59,33.01,614.00,679.00,694.00,719.00,8.500000e+02
open_acc,2260639.0,11.61,5.64,0.00,8.00,11.00,14.00,1.010000e+02
pub_rec,2260639.0,0.20,0.57,0.00,0.00,0.00,0.00,8.600000e+01


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,...,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,open_acc,pub_rec,revol_bal,revol_util,issue_month
1832791,35000.0,60 months,21.98,966.27,E,9 years,RENT,140000.0,Verified,Jun-2013,...,16.55,1.0,Jun-1996,680.0,684.0,25.0,0.0,35106.0,25.2,2013-06-01
1380750,30000.0,36 months,6.46,918.93,A,2 years,RENT,77000.0,Source Verified,Dec-2018,...,14.03,0.0,May-1989,720.0,724.0,5.0,0.0,17643.0,60.8,2018-12-01
560754,13600.0,36 months,12.62,455.76,C,3 years,OWN,40000.0,Source Verified,Sep-2017,...,20.76,3.0,Apr-2003,665.0,669.0,20.0,0.0,7417.0,72.7,2017-09-01


Время профилирования: 11.4 c
FRED: безработица по штатам (XML)
Строк × колонок: (8568, 3) | память, МБ: 0.6


...

Время профилирования: 0.0 c
CFPB: метаданные жалоб (текст)
Строк × колонок: (291574, 12) | память, МБ: 178.8


,dtype,пропуски_%,уникальных
Date received,datetime64[ns],0.00,1138
Product,object,0.00,18
Sub-product,object,17.89,74
Issue,object,0.00,160
Sub-issue,object,34.58,215
Company,object,0.00,3712
State,object,0.39,63
Submitted via,object,0.00,1
Date sent to company,datetime64[ns],0.00,1204
Company response to consumer,object,0.00,5


,count,mean,std,min,25%,50%,75%,max
Complaint ID,291574.0,2254331.85,474883.37,1290155.0,1862759.25,2343453.0,2687373.75,2892274.0


,Date received,Product,Sub-product,Issue,Sub-issue,Company,State,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
273473,2018-03-10,Debt collection,Medical debt,Attempts to collect debt not owed,Debt is not yours,"Commonwealth Financial Systems, Inc.",AK,Web,2018-03-10,Closed with explanation,Yes,2839467
203072,2017-08-30,Credit card or prepaid card,Store credit card,Problem with a credit reporting company's inve...,Their investigation did not fix an error on yo...,SYNCHRONY FINANCIAL,CA,Web,2017-08-30,Closed with non-monetary relief,Yes,2654750
151766,2015-04-25,Credit reporting,NaN,Credit reporting company's investigation,No notice of investigation status/result,Experian Information Solutions Inc.,CO,Web,2015-04-25,Closed with non-monetary relief,Yes,1347252


Время профилирования: 1.2 c


In [16]:
coverage = pd.DataFrame({
    "источник": ["Lending Club", "FRED безработица (XML)", "FRED ставка ФРС (JSON)", "CFPB жалобы с текстом"],
    "начало": [lc["issue_month"].min(), df_unemp["date"].min(), ff["date"].min(), df_txt["Date received"].min()],
    "конец":  [lc["issue_month"].max(), df_unemp["date"].max(), ff["date"].max(), df_txt["Date received"].max()],
    "строк":  [len(lc), len(df_unemp), len(ff), len(df_txt)],
})
coverage["начало"] = coverage["начало"].dt.date
coverage["конец"] = coverage["конец"].dt.date
display(coverage)

,источник,начало,конец,строк
0,Lending Club,2007-06-01,2018-12-01,2260668
1,FRED безработица (XML),2006-01-01,2019-12-01,8568
2,FRED ставка ФРС (JSON),2005-01-01,2019-12-01,180
3,CFPB жалобы с текстом,2015-03-19,2018-04-30,291574


### 4.2	Очистка, выбор общего периода, приведение ключей

In [19]:
import numpy as np, pandas as pd, time
from IPython.display import display

t0 = time.perf_counter()
STATES51 = sorted(df_unemp["state"].unique())            # 50 штатов + DC
P_START, P_END = pd.Timestamp("2015-03-01"), pd.Timestamp("2018-04-01")

log = []
def step(name, df, source):
    log.append({"источник": source, "шаг": name, "строк": len(df)})

lc = lc.copy()
step("исходно", lc, "Lending Club")
lc = lc[lc["addr_state"].isin(STATES51)]
step("только 50 штатов и DC", lc, "Lending Club")
lc = lc[lc["issue_month"].between(P_START, P_END)]
step("общий период", lc, "Lending Club")

# целевая переменная (кредиты старого формата имеют отдельные названия статусов)
status_map = {"Fully Paid": 0, "Charged Off": 1, "Default": 1,
              "Does not meet the credit policy. Status:Fully Paid": 0,
              "Does not meet the credit policy. Status:Charged Off": 1}
lc["default"] = lc["loan_status"].map(status_map)

# приведение типов
lc["term_m"] = lc["term"].astype(str).str.extract(r"(\d+)")[0].astype(int)
for c in ["int_rate", "revol_util"]:
    lc[c] = pd.to_numeric(lc[c].astype(str).str.rstrip("%"), errors="coerce")
emp = lc["emp_length"].str.replace("< 1 year", "0 years", regex=False)
lc["emp_len_y"] = pd.to_numeric(emp.str.extract(r"(\d+)")[0])
lc["emp_len_missing"] = lc["emp_length"].isna().astype(int)
lc["grade_num"] = lc["grade"].map({g: i for i, g in enumerate("ABCDEFG", 1)})

# производные признаки
lc["fico_mean"] = (lc["fico_range_low"] + lc["fico_range_high"]) / 2
ecl = pd.to_datetime(lc["earliest_cr_line"], format="%b-%Y", errors="coerce")
lc["credit_age_y"] = (lc["issue_month"] - ecl).dt.days / 365.25

# выбросы: флаг и обрезка, сырые значения сохраняются
lc["dti_invalid"] = ((lc["dti"] < 0) | (lc["dti"] > 100)).astype(int)
lc["dti_clean"] = lc["dti"].where(lc["dti_invalid"] == 0)
cap = lc["annual_inc"].quantile(0.999)
lc["log_income"] = np.log1p(lc["annual_inc"].clip(upper=cap))

# ЦЕНЗУРИРОВАНИЕ: доля кредитов с известным исходом по году выдачи и сроку
lc["observed"] = lc["default"].notna()
cens = (lc.assign(year=lc["issue_month"].dt.year)
          .pivot_table(index="year", columns="term_m", values="observed", aggfunc="mean")
          .mul(100).round(1))
print("Доля кредитов с известным исходом, %:")
display(cens)

lc_model = lc[lc["observed"]].copy()
lc_model["default"] = lc_model["default"].astype(int)
step("только завершённые кредиты", lc_model, "Lending Club")
print(f"Очистка Lending Club: {time.perf_counter() - t0:.1f} c")
print(f"Доля дефолтов в выборке: {lc_model['default'].mean():.1%}")
print("dti вне диапазона:", int(lc["dti_invalid"].sum()), "| emp_length пропущен:", int(lc["emp_len_missing"].sum()))

Доля кредитов с известным исходом, %:


term_m,36,60
year,,
2015,99.9,66.0
2016,71.8,54.8
2017,40.1,33.1
2018,20.4,17.5


Очистка Lending Club: 14.0 c
Доля дефолтов в выборке: 21.9%
dti вне диапазона: 1497 | emp_length пропущен: 93485


In [21]:
t0 = time.perf_counter()

# --- FRED XML: безработица ---
u = df_unemp.copy()
assert u.duplicated(["state", "date"]).sum() == 0, "дубликаты штат-месяц"
print("Месяцев на штат (мин/макс):", u.groupby("state").size().min(), u.groupby("state").size().max())
u = u.sort_values(["state", "date"])
g = u.groupby("state")["unemployment"]
u["ur_chg_3m"] = g.diff(3)
u["ur_chg_12m"] = g.diff(12)
u["ur_ma_3m"] = g.transform(lambda s: s.rolling(3).mean())     # считаем до обрезки периода
u = u[u["date"].between("2014-01-01", "2018-04-01")]           # запас для лагов
REGION = {**dict.fromkeys(["CT","ME","MA","NH","RI","VT","NJ","NY","PA"], "Northeast"),
          **dict.fromkeys(["IL","IN","MI","OH","WI","IA","KS","MN","MO","NE","ND","SD"], "Midwest"),
          **dict.fromkeys(["DE","DC","FL","GA","MD","NC","SC","VA","WV","AL","KY","MS","TN","AR","LA","OK","TX"], "South"),
          **dict.fromkeys(["AZ","CO","ID","MT","NV","NM","UT","WY","AK","CA","HI","OR","WA"], "West")}
u["region"] = u["state"].map(REGION)
assert u["region"].notna().all()
step("безработица: период 2014-01..2018-04", u, "FRED XML")

# --- FRED JSON: ставка ---
f = ff[ff["date"].between("2014-01-01", "2018-04-01")].copy()
f["rate_regime"] = pd.cut(f["chg_12m"], [-np.inf, -0.25, 0.25, np.inf],
                          labels=["снижение", "стабильно", "рост"])
step("ставка: период 2014-01..2018-04", f, "FRED JSON")

# --- CFPB: жалобы ---
c = df_txt.rename(columns={"Complaint ID": "complaint_id", "Date received": "date_received",
        "Date sent to company": "date_sent", "Product": "product", "Issue": "issue",
        "State": "state", "Company": "company", "Submitted via": "submitted_via",
        "Company response to consumer": "company_response", "Timely response?": "timely",
        "Consumer complaint narrative": "narrative"}).copy()
step("исходно (с текстом)", c, "CFPB")
c = c.drop_duplicates("complaint_id")
step("без дубликатов по Complaint ID", c, "CFPB")
c = c[c["date_received"].between("2015-03-01", "2018-04-30")]
step("общий период", c, "CFPB")
c = c[c["state"].isin(STATES51)]
step("только 50 штатов и DC", c, "CFPB")

# очистка текста: маски XXXX, пробелы
c["n_masks"] = c["narrative"].str.count(r"X{2,}")
c["text"] = (c["narrative"].str.replace(r"X{2,}", " ", regex=True)
                            .str.replace(r"\s+", " ", regex=True).str.strip())
c["n_words"] = c["text"].str.split().str.len()
c = c[c["n_words"] >= 5]
step("тексты не короче 5 слов", c, "CFPB")

# объединение переименованных продуктов
RULES = [("debt collection", "debt_collection"), ("credit reporting", "credit_reporting"),
         ("credit card", "credit_card"), ("payday", "consumer_loan"), ("consumer loan", "consumer_loan"),
         ("student loan", "student_loan"), ("vehicle", "vehicle_loan"), ("mortgage", "mortgage")]
def group(p):
    p = str(p).lower()
    return next((g for k, g in RULES if k in p), "other")
c["product_group"] = c["product"].map(group)
c["is_debt_product"] = c["product_group"].isin(
    ["debt_collection", "credit_card", "consumer_loan", "student_loan", "vehicle_loan"]).astype(int)
print(c["product_group"].value_counts())

c["response_days"] = (c["date_sent"] - c["date_received"]).dt.days
c.loc[c["response_days"] < 0, "response_days"] = np.nan
print(f"Очистка FRED и CFPB: {time.perf_counter() - t0:.1f} c")

display(pd.DataFrame(log))

Месяцев на штат (мин/макс): 168 168


C:\Users\user\AppData\Local\Temp\ipykernel_17324\1929192613.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  u["region"] = u["state"].map(REGION)


product_group
credit_reporting    84934
debt_collection     66264
mortgage            45018
credit_card         30629
other               28136
student_loan        17417
consumer_loan       13643
vehicle_loan         3198
Name: count, dtype: int64
Очистка FRED и CFPB: 52.0 c


,источник,шаг,строк
0,Lending Club,исходно,2260668
1,Lending Club,только 50 штатов и DC,2260668
2,Lending Club,общий период,1390996
3,Lending Club,только завершённые кредиты,813554
4,FRED XML,безработица: период 2014-01..2018-04,2652
5,FRED JSON,ставка: период 2014-01..2018-04,52
6,CFPB,исходно (с текстом),291574
7,CFPB,без дубликатов по Complaint ID,291574
8,CFPB,общий период,291574
9,CFPB,только 50 штатов и DC,289559


In [23]:
# единые ключи: state (верхний регистр) и month (начало месяца)
lc_model["state"] = lc_model["addr_state"].str.upper().str.strip()
lc_model["month"] = lc_model["issue_month"]
lc_model["month_lag1"] = lc_model["month"] - pd.DateOffset(months=1)   # окружение за предыдущий месяц

u = u.rename(columns={"date": "month"})
f = f.rename(columns={"date": "month"})
c["state"] = c["state"].str.upper().str.strip()
c["month"] = c["date_received"].dt.to_period("M").dt.to_timestamp()

# проверка: какая доля кредитов найдёт пару в каждом источнике
chk_u = lc_model.merge(u[["state", "month"]], left_on=["state", "month_lag1"],
                       right_on=["state", "month"], how="left", indicator=True, suffixes=("", "_u"))
chk_f = lc_model.merge(f[["month"]], left_on="month_lag1", right_on="month",
                       how="left", indicator=True, suffixes=("", "_f"))
pairs = c.groupby(["state", "month"]).size().rename("n").reset_index()
chk_c = lc_model.merge(pairs, left_on=["state", "month_lag1"], right_on=["state", "month"],
                       how="left", suffixes=("", "_c"))

print("Покрытие безработицей, %:", round((chk_u["_merge"] == "both").mean() * 100, 2))
print("Покрытие ставкой ФРС, %:", round((chk_f["_merge"] == "both").mean() * 100, 2))
print("Есть жалобы с текстом в штате за предыдущий месяц, %:", round(chk_c["n"].notna().mean() * 100, 2))
print("Медиана жалоб в паре штат-месяц:", pairs["n"].median())
print("Пар штат-месяц с < 5 жалобами, %:", round((pairs["n"] < 5).mean() * 100, 1))

# сохранение очищенных данных
import os; os.makedirs("data/clean", exist_ok=True)
lc_model.to_parquet("data/clean/lc_model.parquet"); lc.to_parquet("data/clean/lc_all_period.parquet")
u.to_parquet("data/clean/unemp.parquet"); f.to_parquet("data/clean/ff.parquet")
c.to_parquet("data/clean/cfpb_clean.parquet")

Покрытие безработицей, %: 100.0
Покрытие ставкой ФРС, %: 100.0
Есть жалобы с текстом в штате за предыдущий месяц, %: 97.18
Медиана жалоб в паре штат-месяц: 71.0
Пар штат-месяц с < 5 жалобами, %: 1.5


Общий период проекта задан как март 2015 – апрель 2018 (38 месяцев): его определяет CFPB, у которого тексты жалоб есть только с 19.03.2015, а выгрузка заканчивается 30.04.2018. Доля дефолтов среди кредитов с известным исходом в этом периоде составила 21,9 %, то есть классы несбалансированы (примерно 1 к 3,5).

Цензурирование остаётся главным ограничением. Исход известен у 99,9 % 36-месячных кредитов 2015 года, но для остальных когорт доля резко падает: 2016 год даёт 71,8 % (36 мес.) и 54,8 % (60 мес.), 2017 год 40,1 % и 33,1 %, 2018 год 20,4 % и 17,5 %. У 60-месячных кредитов 2015 года исход известен лишь в 66,0 % случаев. Среди поздних когорт завершённые кредиты смещены в сторону рано погашенных и рано списанных, поэтому доля дефолтов для них оценивается со смещением. Поэтому основная модель строится на кредитах 2015–2016 годов, а полную выборку используем как проверку устойчивости результата. 36- и 60-месячные кредиты нужно сравнивать с осторожностью (гипотеза H1.6).

Очистка Lending Club. emp_length переведён в число лет, пропущено 93 485 значений (добавлен флаг пропуска), dti вне диапазона 0–100 отмечен флагом у 1 497 кредитов. Пропуски не заполнялись, чтобы заполнить их внутри пайплайна модели без утечки данных. Выбросы дохода обрезаны, сырые значения сохранены.

Очистка FRED. Безработица: 168 месяцев на каждый штат, дубликатов нет. Добавлены изменения за 3 и 12 месяцев, скользящее среднее и регион переписи.

Очистка CFPB. После удаления дубликатов, фильтрации по периоду и штатам, очистки масок XXXX и удаления коротких текстов осталось 289 239 жалоб. Продукты объединены в группы: кредитный рейтинг 29,4 %, взыскание долгов 22,9 %, ипотека 15,6 %, кредитные карты 10,6 %, студенческие кредиты 6,0 %, потребительские кредиты 4,7 %, автокредиты 1,1 %, прочие 9,7 %. К долговым продуктам относятся 131 151 жалоба (45,3 %). Ипотека выходит за границы исследования потребительских кредитов.

Слияние по ключам state + month с лагом в один месяц (данные внешней среды берутся за месяц, предшествующий выдаче, чтобы исключить утечку из будущего). Безработицей и ставкой ФРС покрыто 100 % кредитов, жалобы с текстом в том же штате за предыдущий месяц есть у 97,2 % кредитов. Медиана составляет 71 жалобу на пару «штат × месяц», а пар с менее чем пятью жалобами 1,5 %. Но этот показатель считался только по существующим парам, а оставшиеся 2,8 % кредитов вообще без жалоб в штате за предыдущий месяц. Для таких кредитов признаки жалоб будут пропущены, и это нужно учесть при моделировании (отдельный флаг или нулевое значение числа жалоб).

Время: очистка Lending Club заняла 14,0 с, очистка FRED и CFPB 52,0 с.

### 4.3	Аннотирование и векторизация текста CFPB (тональность, TF-IDF, темы)

In [26]:
!pip install vaderSentiment scikit-learn pyarrow
import time, numpy as np, pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

T43 = {}                                     # замеры времени для п. 2.5
sia = SentimentIntensityAnalyzer()
score = lambda t: sia.polarity_scores(t)["compound"]

# пробный замер на 5 000 текстов, чтобы оценить время полного прогона
t0 = time.perf_counter()
_ = c["text"].iloc[:5000].map(score)
per_doc = (time.perf_counter() - t0) / 5000
print(f"~{per_doc * len(c) / 60:.1f} мин на полный прогон")

~25.0 мин на полный прогон


In [28]:
t0 = time.perf_counter()
c["sentiment"] = c["text"].map(score)
T43["Тональность VADER"] = time.perf_counter() - t0
c["is_negative"] = (c["sentiment"] <= -0.05).astype(int)

print(f"VADER: {T43['Тональность VADER'] / 60:.1f} мин")
print(c["sentiment"].describe().round(3))
print("Доля негативных, %:", round(c["is_negative"].mean() * 100, 1))
print(c.groupby("product_group")["sentiment"].mean().round(3).sort_values())

VADER: 18.0 мин
count    289239.000
mean         -0.033
std           0.687
min          -1.000
25%          -0.718
50%          -0.034
75%           0.650
max           1.000
Name: sentiment, dtype: float64
Доля негативных, %: 49.8
product_group
debt_collection    -0.266
other              -0.144
mortgage           -0.024
vehicle_loan       -0.009
consumer_loan       0.006
student_loan        0.041
credit_reporting    0.100
credit_card         0.132
Name: sentiment, dtype: float64


Тональность (VADER). Расчёт занял 18,0 мин (оценка по пробному замеру была 25 мин). Среднее значение −0,03 при стандартном отклонении 0,69, негативными оказались 49,8 % жалоб. Квартили (−0,72 и +0,65) показывают, что распределение двугорбое: жалобы либо сильно негативные, либо сильно позитивные. VADER рассчитан на короткие тексты, а средняя жалоба занимает 196 слов, поэтому оценки у длинных текстов уходят к крайним значениям. Среднее при таком распределении малоинформативно, и лучше работать с долей негативных. Между продуктами различия заметные: самая негативная группа взыскание долгов (−0,27), самые «позитивные» кредитные карты (+0,13) и кредитный рейтинг (+0,10). Позитивные значения у них, скорее всего, вызваны вежливой лексикой («thank you»), а не реальной удовлетворённостью, поэтому тональность нельзя трактовать как степень недовольства

In [30]:
t0 = time.perf_counter()
tl = c["text"].str.lower()
RULES_KW = {
    "kw_harass":   r"harass|threat|abusive|call(?:s|ed|ing)? me|phone calls|cease and desist|rude",
    "kw_not_mine": r"not my debt|do not owe|don't owe|never owed|not mine|never had an account|wrong person",
    "kw_fraud":    r"identity theft|fraud|stolen|scam",
    "kw_legal":    r"lawsuit|\bsue[ds]?\b|court|attorney|lawyer|garnish",
    "kw_hardship": r"lost my job|unemploy|laid off|medical bill|can't afford|cannot afford|hardship|disabled|fixed income",
}
for name, pat in RULES_KW.items():
    c[name] = tl.str.contains(pat, regex=True).astype(int)
T43["Правила (regex)"] = time.perf_counter() - t0

print((c[list(RULES_KW)].mean() * 100).round(1).rename("доля жалоб, %"))
print(c.groupby("product_group")[list(RULES_KW)].mean().mul(100).round(1))

kw_harass      15.6
kw_not_mine     4.5
kw_fraud       16.9
kw_legal       13.7
kw_hardship     4.3
Name: доля жалоб, %, dtype: float64
                  kw_harass  kw_not_mine  kw_fraud  kw_legal  kw_hardship
product_group                                                            
consumer_loan          23.5          1.4      11.5      12.3          5.9
credit_card            12.4          1.8      22.6       6.2          2.9
credit_reporting        4.3          5.3      20.0      11.2          2.0
debt_collection        29.3         10.3      15.2      19.0          4.4
mortgage               17.6          0.8      10.1      20.6          7.9
other                  11.6          1.2      25.2       7.4          2.5
student_loan           18.5          0.8       8.0      12.9         10.3
vehicle_loan           22.0          1.2      11.4      12.3          6.1


Правила-аннотации. Мошенничество упоминается в 16,9 % жалоб, давление коллекторов в 15,6 %, судебные угрозы в 13,7 %, оспаривание долга в 4,5 %, финансовые трудности в 4,3 %. Профиль по продуктам логичен: в жалобах на взыскание долгов давление коллекторов встречается в 29,3 % случаев, а оспаривание долга в 10,3 %, то есть в 2–10 раз чаще, чем в других группах. Финансовые трудности чаще всего упоминаются в жалобах на студенческие кредиты (10,3 %) и ипотеку (7,9 %). Эти данные подтверждают, что правила улавливают смысл, но они грубые и дают оценку снизу.

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

rng = np.random.RandomState(42)
idx_fit = rng.choice(len(c), size=100_000, replace=False)

t0 = time.perf_counter()
tfidf = TfidfVectorizer(max_features=30_000, min_df=50, max_df=0.5, ngram_range=(1, 2),
                        stop_words="english", sublinear_tf=True, dtype=np.float32)
tfidf.fit(c["text"].iloc[idx_fit])
T43["TF-IDF: обучение словаря (100 тыс.)"] = time.perf_counter() - t0

t0 = time.perf_counter()
X = tfidf.transform(c["text"])
T43["TF-IDF: преобразование всех жалоб"] = time.perf_counter() - t0
print("Матрица:", X.shape, "| ненулевых, %:", round(X.nnz / (X.shape[0] * X.shape[1]) * 100, 3))
print("Размер, МБ:", round((X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6, 0))

Матрица: (289239, 20897) | ненулевых, %: 0.346
Размер, МБ: 169.0


Векторизация TF-IDF. Словарь обучен на 100 000 жалоб за 46,3 с, преобразование всех текстов заняло 76,3 с. Итоговая матрица 289 239 × 20 897 (после отсечения редких слов с min_df=50 осталось 20 897 признаков вместо заданных 30 000), заполнена на 0,346 %, занимает 169 МБ в разреженном формате. Плотная матрица заняла бы около 24 ГБ (float32), то есть разреженность даёт выигрыш примерно в 140 раз.

In [41]:
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

K = 10
terms = tfidf.get_feature_names_out()

t0 = time.perf_counter()
nmf = NMF(n_components=K, init="nndsvda", random_state=42, max_iter=300)
nmf.fit(X[idx_fit])
T43["NMF: обучение (100 тыс.)"] = time.perf_counter() - t0

t0 = time.perf_counter()
W = nmf.transform(X).astype(np.float32)
T43["NMF: преобразование всех жалоб"] = time.perf_counter() - t0
c["topic"] = W.argmax(axis=1)
Wn = W / (W.sum(axis=1, keepdims=True) + 1e-9)          # доли тем в жалобе

for k in range(K):
    top = [terms[i] for i in nmf.components_[k].argsort()[-10:][::-1]]
    print(f"Тема {k} ({(c['topic'] == k).mean() * 100:.1f} %):", ", ".join(top))

Тема 0 (8.7 %): phone, called, told, number, calls, said, asked, calling, times, did
Тема 1 (12.2 %): report, credit report, removed, account, reporting, remove, reported, experian, removed credit, showing
Тема 2 (13.2 %): debt, collection, collect, agency, collection agency, owe, company, collect debt, debt collection, collector
Тема 3 (20.7 %): loan, payment, payments, mortgage, loans, late, pay, 00, month, home
Тема 4 (13.0 %): account, bank, 00, money, check, checking, america, bank america, funds, checking account
Тема 5 (9.8 %): reporting, information, credit reporting, file, letter, fcra, verified, request, accounts, sent
Тема 6 (4.0 %): inquiries, inquiry, hard, inquiries credit, unauthorized, hard inquiries, authorize, did authorize, hard inquiry, did
Тема 7 (5.8 %): identity, theft, identity theft, equifax, victim, victim identity, security, police, information, breach
Тема 8 (3.1 %): wells, fargo, wells fargo, mortgage, home, fargo bank, modification, sale, foreclosure, prop

Темы (NMF). Десять тем интерпретируемы. Названия, предложенные по ключевым словам:

№	Доля	Предлагаемое название
0	8,7 %	Телефонные звонки и контакты
1	12,2 %	Удаление записей из кредитного отчёта
2	13,2 %	Взыскание долгов, коллекторы
3	20,7 %	Платежи по кредитам и ипотеке
4	13,0 %	Банковские счета и средства
5	9,8 %	Проверка данных в кредитных бюро (FCRA)
6	4,0 %	Запросы в кредитную историю (hard inquiries)
7	5,8 %	Кража личности и утечки данных
8	3,1 %	Ипотека Wells Fargo, взыскание залога
9	9,4 %	Кредитные карты и начисления

In [36]:
t0 = time.perf_counter()
topic_cols = [f"topic_{k}" for k in range(K)]
c[topic_cols] = Wn

agg = (c.groupby(["state", "month"])
         .agg(n_complaints=("complaint_id", "size"),
              n_debt=("is_debt_product", "sum"),
              mean_sentiment=("sentiment", "mean"),
              share_negative=("is_negative", "mean"),
              mean_words=("n_words", "mean"),
              mean_response_days=("response_days", "mean"),
              share_not_timely=("timely", lambda s: (s == "No").mean()),
              share_harass=("kw_harass", "mean"),
              share_not_mine=("kw_not_mine", "mean"),
              share_fraud=("kw_fraud", "mean"),
              share_legal=("kw_legal", "mean"),
              share_hardship=("kw_hardship", "mean"),
              **{f"share_{t}": (t, "mean") for t in topic_cols})
         .reset_index())

# нормировка без данных о населении: отклонение от обычного уровня самого штата (z-оценка)
g = agg.groupby("state")["n_complaints"]
agg["complaints_z"] = (agg["n_complaints"] - g.transform("mean")) / g.transform("std")
agg["log_complaints"] = np.log1p(agg["n_complaints"])
T43["Агрегация штат×месяц"] = time.perf_counter() - t0

os.makedirs("data/clean", exist_ok=True)
agg.to_parquet("data/clean/cfpb_state_month.parquet")
c.drop(columns=["narrative"]).to_parquet("data/clean/cfpb_annotated.parquet")
pd.DataFrame(W, columns=topic_cols).to_parquet("data/clean/cfpb_topic_vectors.parquet")

print("Агрегат:", agg.shape, "| штатов:", agg["state"].nunique(), "| месяцев:", agg["month"].nunique())
display(agg.head(3))
display(pd.Series(T43).round(1).rename("секунд"))

Агрегат: (1938, 26) | штатов: 51 | месяцев: 38


,state,month,n_complaints,n_debt,mean_sentiment,share_negative,mean_words,mean_response_days,share_not_timely,share_harass,...,share_topic_2,share_topic_3,share_topic_4,share_topic_5,share_topic_6,share_topic_7,share_topic_8,share_topic_9,complaints_z,log_complaints
0,AK,2015-03-01,4,3,0.266375,0.250000,344.500000,2.750000,0.0,0.500000,...,0.181676,0.106304,0.116564,0.060482,0.017659,0.039979,0.002296,0.116171,-1.000907,1.609438
1,AK,2015-04-01,7,5,0.050486,0.428571,99.428571,8.714286,0.0,0.285714,...,0.211188,0.266379,0.059285,0.042973,0.012564,0.023590,0.007607,0.023733,-0.427523,2.079442
2,AK,2015-05-01,5,3,-0.014400,0.600000,121.600000,1.400000,0.0,0.600000,...,0.160110,0.167380,0.091379,0.107741,0.010413,0.013692,0.010447,0.134351,-0.809779,1.791759


Тональность VADER                      1078.2
Правила (regex)                          39.4
TF-IDF: обучение словаря (100 тыс.)      46.3
TF-IDF: преобразование всех жалоб        76.3
NMF: обучение (100 тыс.)                 22.8
NMF: преобразование всех жалоб            1.9
Агрегация штат×месяц                      0.5
Name: секунд, dtype: float64

### 4.4	Агрегаты и слияние в единую таблицу, проверка результата

In [44]:
import numpy as np, pandas as pd, time
from IPython.display import display
T44 = {}

t0 = time.perf_counter()
months = pd.date_range("2015-03-01", "2018-04-01", freq="MS")
grid = pd.MultiIndex.from_product([STATES51, months], names=["state", "month"]).to_frame(index=False)
g = grid.merge(agg, on=["state", "month"], how="left").sort_values(["state", "month"]).reset_index(drop=True)
g[["n_complaints", "n_debt"]] = g[["n_complaints", "n_debt"]].fillna(0)

def rsum(col):
    return g.groupby("state")[col].transform(lambda x: x.rolling(3, min_periods=1).sum())

g["n_3m"] = rsum("n_complaints")
g["debt_3m"] = rsum("n_debt")
SHARE = {"sentiment": "mean_sentiment", "neg": "share_negative", "harass": "share_harass",
         "notmine": "share_not_mine", "fraud": "share_fraud", "legal": "share_legal",
         "hardship": "share_hardship", "not_timely": "share_not_timely"}
for k, col in SHARE.items():
    g["_w"] = g[col].fillna(0) * g["n_complaints"]
    g[f"{k}_3m"] = rsum("_w") / g["n_3m"].replace(0, np.nan)
g = g.drop(columns="_w")
g["debt_share_3m"] = g["debt_3m"] / g["n_3m"].replace(0, np.nan)
g["n_3m_log"] = np.log1p(g["n_3m"])

keep = ["state", "month", "n_3m", "n_3m_log", "debt_share_3m"] + [f"{k}_3m" for k in SHARE]
cf = g[keep].rename(columns={c: f"cf_{c}" for c in keep if c not in ["state", "month"]})
cf = cf.rename(columns={"month": "month_lag1"})
T44["Сглаженные агрегаты жалоб"] = time.perf_counter() - t0
print("Сетка:", g.shape, "| пар штат-месяц без жалоб, %:", round((g["n_complaints"] == 0).mean() * 100, 1))

Сетка: (1938, 38) | пар штат-месяц без жалоб, %: 0.0


In [46]:
t0 = time.perf_counter()
lc_model = lc_model.reset_index(drop=True)
lc_model["loan_id"] = lc_model.index

u2 = (u[["state", "month", "unemployment", "ur_chg_3m", "ur_chg_12m", "ur_ma_3m", "region"]]
        .rename(columns={"month": "month_lag1", "unemployment": "ur_level"}))
f2 = (f[["month", "fed_funds_rate", "chg_3m", "chg_12m", "ma_6m", "rate_regime"]]
        .rename(columns={"month": "month_lag1", "fed_funds_rate": "ff_rate", "chg_3m": "ff_chg_3m",
                         "chg_12m": "ff_chg_12m", "ma_6m": "ff_ma_6m", "rate_regime": "ff_regime"}))

base_cols = ["loan_id", "state", "month", "month_lag1", "default", "loan_amnt", "term_m", "int_rate",
             "installment", "grade", "grade_num", "emp_len_y", "emp_len_missing", "home_ownership",
             "annual_inc", "log_income", "verification_status", "purpose", "dti_clean", "dti_invalid",
             "delinq_2yrs", "open_acc", "pub_rec", "revol_bal", "revol_util", "fico_mean", "credit_age_y"]
n0 = len(lc_model)
m = (lc_model[base_cols]
     .merge(u2, on=["state", "month_lag1"], how="left", validate="many_to_one")
     .merge(f2, on="month_lag1", how="left", validate="many_to_one")
     .merge(cf, on=["state", "month_lag1"], how="left", validate="many_to_one"))
T44["Слияние pandas (merge)"] = time.perf_counter() - t0

assert len(m) == n0, "число строк изменилось"
assert m["loan_id"].is_unique
assert (m["month_lag1"] < m["month"]).all(), "внешние признаки из будущего"
m["issue_year"] = m["month"].dt.year
m["main_sample"] = (m["month"] < "2017-01-01").astype(int)     # когорты 2015-2016
m["has_cfpb_window"] = m["cf_n_3m"].notna().astype(int)

print("Итоговая таблица:", m.shape, "| время:", round(T44["Слияние pandas (merge)"], 1), "c")
ext = [c for c in m.columns if c.startswith(("ur_", "ff_", "cf_"))]
display((m[ext].isna().mean() * 100).round(2).rename("пропуски, %"))
print("Основная выборка (2015-2016):", int(m["main_sample"].sum()),
      "| доля дефолтов:", round(m.loc[m["main_sample"] == 1, "default"].mean() * 100, 1), "%")
display(m.sample(3, random_state=1).T)

Итоговая таблица: (813554, 51) | время: 2.6 c


ur_level            0.00
ur_chg_3m           0.00
ur_chg_12m          0.00
ur_ma_3m            0.00
ff_rate             0.00
ff_chg_3m           0.00
ff_chg_12m          0.00
ff_ma_6m            0.00
ff_regime           0.00
cf_n_3m             2.82
cf_n_3m_log         2.82
cf_debt_share_3m    2.82
cf_sentiment_3m     2.82
cf_neg_3m           2.82
cf_harass_3m        2.82
cf_notmine_3m       2.82
cf_fraud_3m         2.82
cf_legal_3m         2.82
cf_hardship_3m      2.82
cf_not_timely_3m    2.82
Name: пропуски, %, dtype: float64

Основная выборка (2015-2016): 614779 | доля дефолтов: 21.7 %


,758783,70987,619112
loan_id,758783,70987,619112
state,IN,MA,CA
month,2016-12-01 00:00:00,2015-10-01 00:00:00,2017-03-01 00:00:00
month_lag1,2016-11-01 00:00:00,2015-09-01 00:00:00,2017-02-01 00:00:00
default,1,0,0
loan_amnt,3000.0,7000.0,6000.0
term_m,36,36,36
int_rate,7.49,7.89,7.24
installment,93.31,219.0,185.93
grade,A,A,A


In [48]:
main = m[m["main_sample"] == 1]
print("Доля дефолтов по грейду, %:\n", main.groupby("grade")["default"].mean().mul(100).round(1).to_string())
print("\nПо сроку, %:\n", main.groupby("term_m")["default"].mean().mul(100).round(1).to_string())
print("\nПо году выдачи, %:\n", main.groupby("issue_year")["default"].mean().mul(100).round(1).to_string())
for col in ["ur_level", "cf_harass_3m"]:
    q = pd.qcut(main[col], 4, duplicates="drop")
    print(f"\nПо квартилям {col}:")
    display(main.groupby(q, observed=True)["default"].agg(["mean", "size"]).round(3))

Доля дефолтов по грейду, %:
 grade
A     6.3
B    14.5
C    24.3
D    34.1
E    43.1
F    52.5
G    56.2

По сроку, %:
 term_m
36    17.3
60    36.6

По году выдачи, %:
 issue_year
2015    20.3
2016    23.3

По квартилям ur_level:


,mean,size
ur_level,,
"(2.6990000000000003, 4.7]",0.212,167653
"(4.7, 5.2]",0.232,155363
"(5.2, 5.7]",0.215,152871
"(5.7, 7.3]",0.209,138892



По квартилям cf_harass_3m:


,mean,size
cf_harass_3m,,
"(-0.001, 0.163]",0.229,148016
"(0.163, 0.182]",0.219,149397
"(0.182, 0.206]",0.210,146790
"(0.206, 1.0]",0.211,147645


In [50]:
mart = (m.groupby(["state", "month"])
          .agg(n_loans=("loan_id", "size"), default_rate=("default", "mean"),
               avg_dti=("dti_clean", "mean"), avg_int_rate=("int_rate", "mean"),
               avg_amount=("loan_amnt", "mean"), avg_fico=("fico_mean", "mean"),
               ur_level=("ur_level", "first"), region=("region", "first"),
               ff_rate=("ff_rate", "first"), cf_n_3m=("cf_n_3m", "first"),
               cf_neg_3m=("cf_neg_3m", "first"), cf_harass_3m=("cf_harass_3m", "first"),
               cf_hardship_3m=("cf_hardship_3m", "first"))
          .reset_index())
mart["small_cell"] = (mart["n_loans"] < 30).astype(int)
print("Витрина:", mart.shape, "| ячеек с < 30 кредитами, %:", round(mart["small_cell"].mean() * 100, 1))

import os; os.makedirs("data/mart", exist_ok=True)
t0 = time.perf_counter()
m.to_parquet("data/mart/loans_merged.parquet", index=False)
mart.to_parquet("data/mart/state_month.parquet", index=False)
cf.to_parquet("data/mart/cfpb_features.parquet", index=False)
T44["Запись parquet"] = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pd.read_parquet("data/mart/loans_merged.parquet")
T44["Чтение parquet"] = time.perf_counter() - t0
print(pd.Series(T44).round(2).rename("секунд"))

Витрина: (1860, 16) | ячеек с < 30 кредитами, %: 5.3
Сглаженные агрегаты жалоб    0.19
Слияние pandas (merge)       2.57
Запись parquet               3.31
Чтение parquet               0.91
Name: секунд, dtype: float64


In [52]:
!pip install duckdb
import duckdb
t0 = time.perf_counter()
con = duckdb.connect()
con.register("lcm", lc_model[base_cols]); con.register("u2", u2)
con.register("f2", f2); con.register("cf", cf)
m_sql = con.execute("""
    SELECT l.*, u.* EXCLUDE (state, month_lag1), f.* EXCLUDE (month_lag1), c.* EXCLUDE (state, month_lag1)
    FROM lcm l
    LEFT JOIN u2 u ON l.state = u.state AND l.month_lag1 = u.month_lag1
    LEFT JOIN f2 f ON l.month_lag1 = f.month_lag1
    LEFT JOIN cf c ON l.state = c.state AND l.month_lag1 = c.month_lag1
""").df()
T44["Слияние DuckDB (SQL)"] = time.perf_counter() - t0
print("Строк:", len(m_sql), "| совпадение сумм ur_level:",
      np.isclose(m_sql["ur_level"].sum(), m["ur_level"].sum()))
print(pd.Series({k: v for k, v in T44.items() if "Слияние" in k}).round(2))

   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
    --------------------------------------- 0.3/15.4 MB ? eta -:--:--
   -- ------------------------------------- 0.8/15.4 MB 2.2 MB/s eta 0:00:07
   -- ------------------------------------- 1.0/15.4 MB 2.3 MB/s eta 0:00:07
   ---- ----------------------------------- 1.6/15.4 MB 2.3 MB/s eta 0:00:06
   ----- ---------------------------------- 2.1/15.4 MB 2.3 MB/s eta 0:00:06
   ------ --------------------------------- 2.6/15.4 MB 2.4 MB/s eta 0:00:06
   -------- ------------------------------- 3.1/15.4 MB 2.4 MB/s eta 0:00:06
   --------- ------------------------------ 3.7/15.4 MB 2.4 MB/s eta 0:00:05
   ---------- ----------------------------- 4.2/15.4 MB 2.4 MB/s eta 0:00:05
   ------------ --------------------------- 4.7/15.4 MB 2.4 MB/s eta 0:00:05
   ------------- -------------------------- 5.2/15.4 MB 2.4 MB/s eta 0:00:05
   --------------- ------------------------ 5.8/15.4 MB 2.4 MB/s eta 0:00:05
   ----------

Итоговая таблица содержит 813 554 кредита и 51 признак. Число строк после слияния совпало с числом завершённых кредитов, дубликатов ключей нет, внешние признаки берутся только за месяцы до выдачи. Слияние pandas заняло 2,57 с, SQL-запрос в DuckDB 2,61 с, контрольные суммы совпали (ur_level). Скорости практически равны, поэтому на таких объёмах выбор метода определяется удобством, а не быстродействием.

Пропуски. В признаках безработицы и ставки ФРС пропусков нет. В признаках жалоб они составляют 2,82 %. Вероятная причина: у кредитов, выданных в марте 2015 года, предыдущий месяц (февраль 2015) лежит до начала текстов CFPB. Это стоит проверить командой m.loc[m["has_cfpb_window"] == 0, "month"].value_counts(). В выводе к 4.2 я писал, что эти 2,8 % кредитов «без жалоб в штате», это было неверно: сетка показала 0,0 % пар «штат × месяц» без жалоб, то есть пропуски появляются только из-за границы периода.

Проверка здравого смысла пройдена. Доля дефолтов монотонно растёт от грейда A (6,3 %) к грейду G (56,2 %). У 60-месячных кредитов она выше, чем у 36-месячных (36,6 % против 17,3 %). По годам выдачи 20,3 % (2015) и 23,3 % (2016). Основная выборка (кредиты 2015–2016 годов) составляет 614 779 кредитов, доля дефолтов 21,7 %.

Первичные сопоставления с внешними факторами не подтверждают гипотезы H1.4 и H1.5. Доля дефолтов по квартилям безработицы почти не меняется и немонотонна (21,2 %; 23,2 %; 21,5 %; 20,9 %). По квартилям жалоб на давление коллекторов она слегка снижается (22,9 % → 21,1 %), то есть направление противоположно ожидаемому. Это ещё не опровержение: в сопоставление смешаны штат, год, состав заёмщиков и размер штата. Проверка с контролем индивидуальных признаков остаётся для п. 2.4.

Витрина «штат × месяц выдачи»: 1 860 строк (меньше 1 938 из-за 78 ячеек без выданных кредитов), из них 5,3 % ячеек содержат менее 30 кредитов и помечены флагом small_cell.

Время: агрегаты жалоб 0,19 с, слияние 2,57 с, запись в parquet 3,31 с, чтение 0,91 с.

## 5.Сделать обоснованный выбор структуры для хранения объединенных больших данных, написать скрипты слияния и обновления разнородных данных. 

### 5.2. Создание и первичная загрузка

In [58]:
import os, time, duckdb, pandas as pd
os.makedirs("data", exist_ok=True)
con = duckdb.connect("data/warehouse.duckdb")

con.execute("""
CREATE TABLE IF NOT EXISTS dim_time(month DATE PRIMARY KEY, year INT, quarter INT, ym VARCHAR);
CREATE TABLE IF NOT EXISTS dim_geo(state VARCHAR PRIMARY KEY, region VARCHAR);
CREATE TABLE IF NOT EXISTS dim_grade(grade VARCHAR PRIMARY KEY, grade_num INT);
CREATE TABLE IF NOT EXISTS dim_purpose(purpose VARCHAR PRIMARY KEY);
CREATE TABLE IF NOT EXISTS dim_topic(topic INT PRIMARY KEY, topic_name VARCHAR);

CREATE TABLE IF NOT EXISTS fact_loan(
  loan_id BIGINT PRIMARY KEY, month DATE, state VARCHAR, grade VARCHAR, purpose VARCHAR,
  term_m INT, is_default INT, loan_amnt DOUBLE, int_rate DOUBLE, installment DOUBLE,
  annual_inc DOUBLE, dti_clean DOUBLE, delinq_2yrs DOUBLE, open_acc DOUBLE, pub_rec DOUBLE,
  revol_bal DOUBLE, revol_util DOUBLE, fico_mean DOUBLE, credit_age_y DOUBLE, emp_len_y DOUBLE,
  home_ownership VARCHAR, verification_status VARCHAR);

CREATE TABLE IF NOT EXISTS fact_unemp(state VARCHAR, month DATE, ur_level DOUBLE, PRIMARY KEY(state, month));
CREATE TABLE IF NOT EXISTS fact_macro(month DATE PRIMARY KEY, ff_rate DOUBLE);

CREATE TABLE IF NOT EXISTS dim_complaint(
  complaint_id BIGINT PRIMARY KEY, month DATE, state VARCHAR, product_group VARCHAR,
  is_debt_product INT, topic INT, sentiment DOUBLE, is_negative INT, n_words INT,
  response_days DOUBLE, timely VARCHAR, kw_harass INT, kw_not_mine INT, kw_fraud INT,
  kw_legal INT, kw_hardship INT);
CREATE TABLE IF NOT EXISTS complaint_text(complaint_id BIGINT PRIMARY KEY, text VARCHAR);

CREATE TABLE IF NOT EXISTS etl_log(run_ts TIMESTAMP, source VARCHAR, rows_in BIGINT,
  rows_new BIGINT, rows_updated BIGINT, seconds DOUBLE);
""")

In [60]:
T5 = {}
t0 = time.perf_counter()
P = "data/clean/"
con.execute("""INSERT INTO dim_time SELECT CAST(m AS DATE), year(m), quarter(m), strftime(m, '%Y-%m')
  FROM (SELECT unnest(generate_series(TIMESTAMP '2014-01-01', TIMESTAMP '2018-12-01', INTERVAL 1 MONTH)) AS m)""")
con.execute(f"INSERT INTO dim_geo SELECT DISTINCT state, region FROM read_parquet('{P}unemp.parquet')")
con.execute("INSERT INTO dim_grade VALUES " + ",".join(f"('{g}',{i})" for i, g in enumerate("ABCDEFG", 1)))
con.execute("INSERT INTO dim_purpose SELECT DISTINCT purpose FROM read_parquet('data/mart/loans_merged.parquet')")
TOPIC_NAMES = {0: "Звонки и контакты", 1: "Удаление записей из отчёта", 2: "Взыскание долгов",
               3: "Платежи по кредитам и ипотеке", 4: "Банковские счета", 5: "Проверка данных в бюро (FCRA)",
               6: "Запросы в кредитную историю", 7: "Кража личности", 8: "Ипотека, взыскание залога", 9: "Кредитные карты"}
con.executemany("INSERT INTO dim_topic VALUES (?, ?)", list(TOPIC_NAMES.items()))

con.execute("""INSERT INTO fact_loan SELECT loan_id, CAST(month AS DATE), state, grade, purpose, term_m,
  CAST("default" AS INT), loan_amnt, int_rate, installment, annual_inc, dti_clean, delinq_2yrs, open_acc,
  pub_rec, revol_bal, revol_util, fico_mean, credit_age_y, emp_len_y, home_ownership, verification_status
  FROM read_parquet('data/mart/loans_merged.parquet')""")
con.execute(f"INSERT INTO fact_unemp SELECT state, CAST(month AS DATE), unemployment FROM read_parquet('{P}unemp.parquet')")
con.execute(f"INSERT INTO fact_macro SELECT CAST(month AS DATE), fed_funds_rate FROM read_parquet('{P}ff.parquet')")
con.execute(f"""INSERT INTO dim_complaint SELECT complaint_id, CAST(month AS DATE), state, product_group,
  is_debt_product, topic, sentiment, is_negative, n_words, response_days, timely,
  kw_harass, kw_not_mine, kw_fraud, kw_legal, kw_hardship FROM read_parquet('{P}cfpb_annotated.parquet')""")
con.execute(f"INSERT INTO complaint_text SELECT complaint_id, text FROM read_parquet('{P}cfpb_annotated.parquet')")
T5["Загрузка хранилища"] = time.perf_counter() - t0

for t in ["dim_time", "dim_geo", "fact_loan", "fact_unemp", "fact_macro", "dim_complaint", "complaint_text"]:
    print(t, con.execute(f"SELECT count(*) FROM {t}").fetchone()[0])
print(f"Загрузка: {T5['Загрузка хранилища']:.1f} c | файл БД, МБ: {os.path.getsize('data/warehouse.duckdb')/1e6:.0f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dim_time 60
dim_geo 51
fact_loan 813554
fact_unemp 2652
fact_macro 52
dim_complaint 289239
complaint_text 289239
Загрузка: 6.4 c | файл БД, МБ: 420


### 5.3. Представления (слияние на стороне хранилища)

In [62]:
con.execute("""
CREATE OR REPLACE VIEW v_unemp AS
SELECT state, month, ur_level,
  ur_level - LAG(ur_level, 3)  OVER w AS ur_chg_3m,
  ur_level - LAG(ur_level, 12) OVER w AS ur_chg_12m,
  AVG(ur_level) OVER (PARTITION BY state ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ur_ma_3m
FROM fact_unemp WINDOW w AS (PARTITION BY state ORDER BY month);

CREATE OR REPLACE VIEW v_macro AS
SELECT month, ff_rate,
  ff_rate - LAG(ff_rate, 3)  OVER w AS ff_chg_3m,
  ff_rate - LAG(ff_rate, 12) OVER w AS ff_chg_12m,
  AVG(ff_rate) OVER (ORDER BY month ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS ff_ma_6m,
  CASE WHEN ff_rate - LAG(ff_rate, 12) OVER w IS NULL THEN NULL
       WHEN ff_rate - LAG(ff_rate, 12) OVER w <= -0.25 THEN 'снижение'
       WHEN ff_rate - LAG(ff_rate, 12) OVER w <= 0.25 THEN 'стабильно' ELSE 'рост' END AS ff_regime
FROM fact_macro WINDOW w AS (ORDER BY month);

CREATE OR REPLACE VIEW v_complaint_month AS
SELECT state, month, count(*) AS n, sum(is_debt_product) AS n_debt, sum(sentiment) AS sent_sum,
  sum(is_negative) AS neg, sum(kw_harass) AS harass, sum(kw_not_mine) AS notmine, sum(kw_fraud) AS fraud,
  sum(kw_legal) AS legal, sum(kw_hardship) AS hardship, sum(CASE WHEN timely = 'No' THEN 1 ELSE 0 END) AS not_timely
FROM dim_complaint GROUP BY state, month;

CREATE OR REPLACE VIEW v_complaint_3m AS
WITH grid AS (
  SELECT g.state, t.month FROM dim_geo g CROSS JOIN dim_time t
  WHERE t.month BETWEEN DATE '2015-03-01' AND (SELECT max(month) FROM dim_complaint)),
m AS (
  SELECT grid.state, grid.month, coalesce(c.n,0) n, coalesce(c.n_debt,0) n_debt, coalesce(c.sent_sum,0) sent_sum,
    coalesce(c.neg,0) neg, coalesce(c.harass,0) harass, coalesce(c.notmine,0) notmine, coalesce(c.fraud,0) fraud,
    coalesce(c.legal,0) legal, coalesce(c.hardship,0) hardship, coalesce(c.not_timely,0) not_timely
  FROM grid LEFT JOIN v_complaint_month c USING (state, month))
SELECT state, month,
  SUM(n) OVER w AS cf_n_3m, ln(1 + SUM(n) OVER w) AS cf_n_3m_log,
  SUM(n_debt) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_debt_share_3m,
  SUM(sent_sum) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_sentiment_3m,
  SUM(neg) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_neg_3m,
  SUM(harass) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_harass_3m,
  SUM(notmine) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_notmine_3m,
  SUM(fraud) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_fraud_3m,
  SUM(legal) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_legal_3m,
  SUM(hardship) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_hardship_3m,
  SUM(not_timely) OVER w / NULLIF(SUM(n) OVER w, 0) AS cf_not_timely_3m
FROM m WINDOW w AS (PARTITION BY state ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW);

CREATE OR REPLACE VIEW v_loans_enriched AS
WITH l AS (SELECT *, CAST(month - INTERVAL 1 MONTH AS DATE) AS month_lag1 FROM fact_loan)
SELECT l.*, g.region, u.ur_level, u.ur_chg_3m, u.ur_chg_12m, u.ur_ma_3m,
  mm.ff_rate, mm.ff_chg_3m, mm.ff_chg_12m, mm.ff_ma_6m, mm.ff_regime,
  c.cf_n_3m, c.cf_n_3m_log, c.cf_debt_share_3m, c.cf_sentiment_3m, c.cf_neg_3m, c.cf_harass_3m,
  c.cf_notmine_3m, c.cf_fraud_3m, c.cf_legal_3m, c.cf_hardship_3m, c.cf_not_timely_3m
FROM l
JOIN dim_geo g USING (state)
LEFT JOIN v_unemp u ON u.state = l.state AND u.month = l.month_lag1
LEFT JOIN v_macro mm ON mm.month = l.month_lag1
LEFT JOIN v_complaint_3m c ON c.state = l.state AND c.month = l.month_lag1;
""")

### 5.4. Скрипты обновления

In [66]:
import requests, xml.etree.ElementTree as ET

def log_run(source, rows_in, n_before, n_after, t0):
    new = n_after - n_before
    con.execute("INSERT INTO etl_log VALUES (now(), ?, ?, ?, ?, ?)",
                [source, rows_in, new, rows_in - new, time.perf_counter() - t0])

def ensure_time(upto):
    con.execute("""INSERT OR IGNORE INTO dim_time SELECT CAST(m AS DATE), year(m), quarter(m), strftime(m,'%Y-%m')
      FROM (SELECT unnest(generate_series(TIMESTAMP '2014-01-01', CAST(? AS TIMESTAMP), INTERVAL 1 MONTH)) AS m)""", [upto])

def count(t): return con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]

# --- 1. FRED XML: безработица по штатам ---
def update_unemp(api_key, months_back=24, end="2019-12-31"):
    t0, n0 = time.perf_counter(), count("fact_unemp")
    last = pd.Timestamp(con.execute("SELECT max(month) FROM fact_unemp").fetchone()[0])
    start = (last - pd.DateOffset(months=months_back)).strftime("%Y-%m-%d")
    rows = []
    for st in [r[0] for r in con.execute("SELECT state FROM dim_geo").fetchall()]:
        r = requests.get("https://api.stlouisfed.org/fred/series/observations", timeout=30,
                         params={"series_id": f"{st}UR", "api_key": api_key, "file_type": "xml",
                                 "observation_start": start, "observation_end": end})
        r.raise_for_status()
        for o in ET.fromstring(r.content).findall("observation"):
            if o.get("value") != ".":
                rows.append((st, o.get("date"), float(o.get("value"))))
        time.sleep(0.6)
    new = pd.DataFrame(rows, columns=["state", "month", "ur_level"])
    if new.empty: return
    ensure_time(new["month"].max())
    con.register("new_u", new)
    con.execute("INSERT OR REPLACE INTO fact_unemp SELECT state, CAST(month AS DATE), ur_level FROM new_u")
    log_run("FRED XML unemployment", len(new), n0, count("fact_unemp"), t0)

# --- 2. FRED JSON: ставка ФРС ---
def update_fedfunds(api_key, months_back=24, end="2019-12-31"):
    t0, n0 = time.perf_counter(), count("fact_macro")
    last = pd.Timestamp(con.execute("SELECT max(month) FROM fact_macro").fetchone()[0])
    start = (last - pd.DateOffset(months=months_back)).strftime("%Y-%m-%d")
    r = requests.get("https://api.stlouisfed.org/fred/series/observations", timeout=30,
                     params={"series_id": "FEDFUNDS", "api_key": api_key, "file_type": "json",
                             "observation_start": start, "observation_end": end})
    r.raise_for_status()
    new = pd.DataFrame(r.json()["observations"])[["date", "value"]].rename(columns={"date": "month"})
    new["ff_rate"] = pd.to_numeric(new["value"], errors="coerce")
    new = new.dropna(subset=["ff_rate"])[["month", "ff_rate"]]
    if new.empty: return
    ensure_time(new["month"].max())
    con.register("new_f", new)
    con.execute("INSERT OR REPLACE INTO fact_macro SELECT CAST(month AS DATE), ff_rate FROM new_f")
    log_run("FRED JSON FEDFUNDS", len(new), n0, count("fact_macro"), t0)

In [68]:
import joblib
os.makedirs("models", exist_ok=True)
joblib.dump(tfidf, "models/tfidf.joblib"); joblib.dump(nmf, "models/nmf.joblib")

['models/nmf.joblib']

In [70]:
# 3. CFPB: новые жалобы из выгрузки CSV 
# Требует в сессии: group(), RULES_KW, sia (VADER) из пунктов 4.2 и 4.3
def annotate_new(df):
    df = df.rename(columns={"Complaint ID": "complaint_id", "Date received": "date_received",
        "Date sent to company": "date_sent", "Product": "product", "State": "state",
        "Company response to consumer": "company_response", "Timely response?": "timely",
        "Consumer complaint narrative": "narrative"})
    df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce")
    df["date_sent"] = pd.to_datetime(df["date_sent"], errors="coerce")
    states = {r[0] for r in con.execute("SELECT state FROM dim_geo").fetchall()}
    df = df[df["state"].isin(states) & df["date_received"].notna()].copy()
    df["text"] = (df["narrative"].str.replace(r"X{2,}", " ", regex=True)
                                  .str.replace(r"\s+", " ", regex=True).str.strip())
    df["n_words"] = df["text"].str.split().str.len()
    df = df[df["n_words"] >= 5].copy()
    df["month"] = df["date_received"].dt.to_period("M").dt.to_timestamp()
    df["product_group"] = df["product"].map(group)
    df["is_debt_product"] = df["product_group"].isin(
        ["debt_collection", "credit_card", "consumer_loan", "student_loan", "vehicle_loan"]).astype(int)
    df["response_days"] = (df["date_sent"] - df["date_received"]).dt.days.where(lambda s: s >= 0)
    tl = df["text"].str.lower()
    for name, pat in RULES_KW.items():
        df[name] = tl.str.contains(pat, regex=True).astype(int)
    df["sentiment"] = df["text"].map(lambda t: sia.polarity_scores(t)["compound"])
    df["is_negative"] = (df["sentiment"] <= -0.05).astype(int)
    tfidf_m, nmf_m = joblib.load("models/tfidf.joblib"), joblib.load("models/nmf.joblib")
    df["topic"] = nmf_m.transform(tfidf_m.transform(df["text"])).argmax(axis=1)
    return df

def update_cfpb(csv_path):
    t0, n0 = time.perf_counter(), count("dim_complaint")
    known = {r[0] for r in con.execute("SELECT complaint_id FROM dim_complaint").fetchall()}
    cols = ["Date received", "Product", "Consumer complaint narrative", "State", "Complaint ID",
            "Date sent to company", "Timely response?"]
    parts = []
    for ch in pd.read_csv(csv_path, usecols=cols, chunksize=200_000, low_memory=False):
        ch = ch.dropna(subset=["Consumer complaint narrative"])
        parts.append(ch[~ch["Complaint ID"].isin(known)])        # только новые
    raw = pd.concat(parts, ignore_index=True)
    if raw.empty:
        print("Новых жалоб нет"); return
    new = annotate_new(raw)
    ensure_time(new["month"].max())
    con.register("new_c", new)
    con.execute("""INSERT OR IGNORE INTO dim_complaint SELECT complaint_id, CAST(month AS DATE), state,
      product_group, is_debt_product, topic, sentiment, is_negative, n_words, response_days, timely,
      kw_harass, kw_not_mine, kw_fraud, kw_legal, kw_hardship FROM new_c""")
    con.execute("INSERT OR IGNORE INTO complaint_text SELECT complaint_id, text FROM new_c")
    log_run("CFPB CSV complaints", len(new), n0, count("dim_complaint"), t0)

In [72]:
def run_update(api_key, cfpb_csv=None):
    t0 = time.perf_counter()
    update_unemp(api_key); update_fedfunds(api_key)
    if cfpb_csv: update_cfpb(cfpb_csv)
    print(f"Обновление: {time.perf_counter() - t0:.1f} c")
    return con.execute("SELECT * FROM etl_log ORDER BY run_ts DESC LIMIT 5").df()

Новые жалобы, ставка и безработица обновляют представления автоматически, без пересчёта. Lending Club в этом цикле не обновляется: он поставляется статичным архивом (2007–2018Q4). При появлении нового квартального файла кредиты добавляются тем же INSERT OR IGNORE. Для этого добавьте в usecols исходное поле id и используйте его вместо суррогатного loan_id, иначе нельзя надёжно отличить новый кредит от уже загруженного.

### 5.5. Проверка и замеры

In [76]:
# 1. Контроль целостности (внешние ключи не объявлены: проверяем запросами)
chk = {
 "кредиты со штатом вне dim_geo":  "SELECT count(*) FROM fact_loan WHERE state NOT IN (SELECT state FROM dim_geo)",
 "кредиты с месяцем вне dim_time": "SELECT count(*) FROM fact_loan WHERE month NOT IN (SELECT month FROM dim_time)",
 "кредиты с неизвестным грейдом":  "SELECT count(*) FROM fact_loan WHERE grade NOT IN (SELECT grade FROM dim_grade)",
 "жалобы с темой вне dim_topic":   "SELECT count(*) FROM dim_complaint WHERE topic NOT IN (SELECT topic FROM dim_topic)",
}
for k, q in chk.items(): print(k, "->", con.execute(q).fetchone()[0])

# 2. Эквивалентность с результатом слияния в pandas (таблица m из 4.4)
v = con.execute("""SELECT count(*), sum(ur_level), avg(cf_n_3m), avg(cf_harass_3m), avg(ff_rate), sum(cf_n_3m IS NULL)
                   FROM v_loans_enriched""").fetchone()
mm = (len(m), m["ur_level"].sum(), m["cf_n_3m"].mean(), m["cf_harass_3m"].mean(), m["ff_rate"].mean(), int(m["cf_n_3m"].isna().sum()))
print("DuckDB:", [round(x, 4) for x in v]); print("pandas:", [round(x, 4) for x in mm])

кредиты со штатом вне dim_geo -> 0
кредиты с месяцем вне dim_time -> 0
кредиты с неизвестным грейдом -> 0
жалобы с темой вне dim_topic -> 0
DuckDB: [813554, 4033854.8, 1016.6602, 0.1771, 0.4233, 22931]
pandas: [813554, 4033854.8, 1016.6602, 0.1771, 0.4233, 22931]


In [78]:
# 3. Замеры: один и тот же запрос «доля дефолтов по штату и году» в четырёх вариантах
q = "SELECT state, year(month) y, avg(is_default) dr, count(*) n FROM fact_loan GROUP BY 1, 2"
R = {}
t0 = time.perf_counter(); df = pd.read_csv("accepted_2007_to_2018Q4.csv", usecols=["addr_state", "issue_d", "loan_status"], low_memory=False)
df["y"] = pd.to_datetime(df["issue_d"], format="%b-%Y").dt.year; _ = df.groupby(["addr_state", "y"]).size()
R["pandas, исходный CSV (3 колонки)"] = time.perf_counter() - t0
t0 = time.perf_counter(); d = pd.read_parquet("data/mart/loans_merged.parquet", columns=["state", "month", "default"])
_ = d.groupby(["state", d["month"].dt.year])["default"].mean(); R["pandas, parquet"] = time.perf_counter() - t0
t0 = time.perf_counter(); _ = duckdb.sql("SELECT state, year(month) y, avg(\"default\") FROM 'data/mart/loans_merged.parquet' GROUP BY 1, 2").df()
R["DuckDB по parquet-файлу"] = time.perf_counter() - t0
t0 = time.perf_counter(); _ = con.execute(q).df(); R["DuckDB, таблица хранилища"] = time.perf_counter() - t0
t0 = time.perf_counter(); _ = con.execute("SELECT count(*), avg(is_default) FROM v_loans_enriched WHERE cf_n_3m IS NOT NULL").df()
R["DuckDB, представление со слиянием (4 источника)"] = time.perf_counter() - t0
print(pd.Series(R).round(2).rename("секунд"))
print("Размер, МБ: CSV исходный", round(os.path.getsize("accepted_2007_to_2018Q4.csv")/1e6),
      "| parquet (слитый)", round(os.path.getsize("data/mart/loans_merged.parquet")/1e6),
      "| БД DuckDB", round(os.path.getsize("data/warehouse.duckdb")/1e6))

pandas, исходный CSV (3 колонки)                   27.85
pandas, parquet                                     0.36
DuckDB по parquet-файлу                             0.19
DuckDB, таблица хранилища                           0.04
DuckDB, представление со слиянием (4 источника)     0.11
Name: секунд, dtype: float64
Размер, МБ: CSV исходный 1675 | parquet (слитый) 37 | БД DuckDB 420


In [80]:
V = "v_loans_enriched"
slice_ = con.execute(f"SELECT grade, avg(is_default) dr, count(*) n FROM {V} WHERE state = 'CA' GROUP BY grade ORDER BY grade").df()
dice_ = con.execute(f"SELECT region, term_m, avg(is_default) dr FROM {V} WHERE year(month) = 2016 AND grade IN ('C','D') GROUP BY 1, 2").df()
rollup = con.execute(f"SELECT year(month) y, quarter(month) q, avg(is_default) dr, count(*) n FROM {V} GROUP BY ROLLUP (1, 2) ORDER BY 1, 2").df()
pivot_ = con.execute(f"PIVOT (SELECT region, grade, is_default FROM {V}) ON grade USING avg(is_default) GROUP BY region").df()
display(slice_); display(dice_); display(rollup.head(8)); display(pivot_.round(3))

,grade,dr,n
0,A,0.063979,20882
1,B,0.153254,34048
2,C,0.251328,33510
3,D,0.341656,15700
4,E,0.418378,7400
5,F,0.511355,2554
6,G,0.538058,762


,region,term_m,dr
0,South,36,0.278391
1,Midwest,60,0.351012
2,West,60,0.311559
3,West,36,0.264612
4,Northeast,36,0.281644
5,Midwest,36,0.266141
6,South,60,0.350769
7,Northeast,60,0.371859


,y,q,dr,n
0,2015,1,0.208408,22931
1,2015,2,0.205778,85942
2,2015,3,0.202433,97904
3,2015,4,0.199004,114897
4,2015,<NA>,0.202528,321674
5,2016,1,0.203611,111934
6,2016,2,0.253459,62949
7,2016,3,0.259579,60891


,region,A,B,C,D,E,F,G
0,Northeast,0.068,0.155,0.262,0.349,0.431,0.517,0.565
1,West,0.061,0.141,0.233,0.325,0.396,0.484,0.490
2,South,0.066,0.150,0.251,0.350,0.440,0.527,0.561
3,Midwest,0.061,0.144,0.246,0.337,0.433,0.521,0.550


Целостность. Контрольные запросы не нашли ни одной записи с несуществующим штатом, месяцем, грейдом или темой (0 нарушений). Результат слияния через SQL-представление полностью совпал с pandas: 813 554 строки, одинаковые суммы и средние, 22 931 пропуск в признаках жалоб (2,82 %). Значит, слияние на стороне хранилища эквивалентно слиянию в ноутбуке, и признаки не нужно пересчитывать вручную.

Скорость. Один и тот же запрос «доля дефолтов по штату и году»:

Вариант	Время, с
pandas, исходный CSV (3 колонки)	27,85
pandas, parquet	0,36
DuckDB по parquet-файлу	0,19
DuckDB, таблица хранилища	0,04
DuckDB, представление со слиянием четырёх источников	0,11

Хранилище быстрее чтения исходного CSV примерно в 700 раз и быстрее pandas по parquet примерно в 9 раз. Слияние четырёх источников на лету стоит всего 0,11 с, поэтому дашборд сможет строить срезы интерактивно. Оговорка для записки: CSV содержит около 150 столбцов, а слитый parquet только 51, так что это сравнение порядков величин, а не чистое сравнение форматов.

Объём. Исходный CSV занимает 1 675 МБ, слитый parquet 37 МБ (колоночное сжатие, но и столбцов меньше), файл DuckDB 420 МБ. Файл DuckDB больше parquet, скорее всего потому, что в нём хранятся и тексты жалоб (около 289 тыс. текстов), и все таблицы, но эту причину я не проверял.

Операции OLAP работают и дают осмысленные результаты:

Срез (Калифорния): доля дефолтов монотонно растёт от грейда A (6,4 %) к G (53,8 %).
Вырезка (2016 год, грейды C и D): 60-месячные кредиты рискованнее 36-месячных во всех регионах (35–37 % против 26–28 %), что поддерживает гипотезу H1.6. Сильнее всего разрыв в Northeast (37,2 % против 28,2 %).
Свёртка (по кварталам): в 2015 году доля стабильна (20,0–20,8 %, итог 20,3 %). В 2016 году она резко растёт: 20,4 % в первом квартале и 25,3 % во втором, при этом число кредитов падает со 111 934 до 62 949. Это не обязательно признак ухудшения качества. Скачок согласуется с цензурированием (в завершённые поздних когорт попадают в основном рано списанные кредиты), но это моё предположение, его нужно проверить отдельно.
Сводная таблица «регион × грейд»: рост по грейду есть во всех регионах. Заметны небольшие, но устойчивые различия внутри грейда: Northeast и South выше, West ниже (например, для грейда D: 34,9 % и 35,0 % против 32,5 %). Разброс между регионами растёт с ростом риска и достигает 7,5 п.п. для грейда G (56,5 % в Northeast против 49,0 % в West). Это описательный результат без контроля других факторов, поэтому как подтверждение H1.2 его использовать нельзя.